# 04. Files, JSON, and explicit data boundaries

**Level:** Beginner

**Prerequisites:** 02-03: containers and exceptions.

[Course index](00_START_HERE.ipynb) | [Previous](03_Python_Control_Functions_and_OOP.ipynb) | [Next](05_HTTP_REST_and_Async.ipynb)

**Topics:** File handling, JSON, Context managers, Data validation.

**Matching script:** `examples\04_Files_JSON_and_Data_Validation.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

A with block closes a file when the block finishes, including on failure. pathlib represents
filesystem paths without manual string concatenation. State encoding explicitly for portable
text files. These examples use a TemporaryDirectory, so they do not overwrite your dataset.

JSON represents objects, arrays, strings, numbers, booleans, and null. Python's json module
converts between JSON text and dictionaries/lists. JSON parsing verifies syntax, not your domain
schema: a decoded value may still lack fields, use an unknown version, or contain invalid units.

Validate external records before training or serving. Require fields that matter, choose a
policy for unknown fields, and report malformed data. Do not use eval to parse a data file, and
do not load untrusted pickle/joblib artifacts: loading executable object formats can execute code.
A public report should allowlist fields rather than dump every internal object attribute.

## Mathematics or data contract

A round trip should preserve the intended public values: decode(encode(record)) == record.

## Important limitation

A successful JSON round trip proves representation, not correctness, privacy, or data provenance.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Safely write, read, and reject a malformed record

**Purpose:** Practice a temporary file plus schema checks.

### Read the code in this order
1. Validate the exact required field set.
2. Write UTF-8 JSON inside a temporary folder.
3. Read the content back using a context manager.
4. Reject a negative latency instead of silently fixing it.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- `raise` reports invalid input or a failed requirement instead of fabricating success.
- `with`/`async with` bounds a resource lifetime or a temporary execution mode.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
invalid record rejected
{'hostname': 'edge-01', 'latency_ms': 12.5}
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

def validate(record):
    if not isinstance(record, dict) or set(record) != {"hostname", "latency_ms"}:
        raise ValueError("unexpected record fields")
    latency = record["latency_ms"]
    if not isinstance(record["hostname"], str) or not record["hostname"].strip():
        raise ValueError("hostname must be nonempty")
    if type(latency) not in (int, float) or not 0 <= latency < float("inf"):
        raise ValueError("finite nonnegative latency required")
    return record

original = validate({"hostname": "edge-01", "latency_ms": 12.5})
with TemporaryDirectory() as folder:
    path = Path(folder) / "sample.json"
    with path.open("w", encoding="utf-8") as handle:
        json.dump(original, handle)
    with path.open(encoding="utf-8") as handle:
        restored = validate(json.load(handle))
    assert restored == original
try:
    validate({"hostname": "edge-01", "latency_ms": -1})
except ValueError:
    print("invalid record rejected")
else:
    raise AssertionError("negative value accepted")
print(restored)

invalid record rejected
{'hostname': 'edge-01', 'latency_ms': 12.5}


### Interpret example 1

The result should demonstrate this purpose: **Practice a temporary file plus schema checks.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Add a schema_version field and reject a version your reader does not understand.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `File handling` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: A round trip should preserve the intended public values: decode(encode(record)) == record.
- Name the boundary of the demonstration: A successful JSON round trip proves representation, not correctness, privacy, or data provenance.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://docs.python.org/3.13/tutorial/`
- `https://docs.python.org/3.13/library/json.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.